# optimizer-init-params-list — worked example 1: Demonstrate generator exhaustion and the list fix

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `optimizer-init-params-list`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

A Python generator can only be iterated once. When an optimizer stores a raw generator as `self.params`, the first `.step()` call exhausts it, and every subsequent call silently does nothing — parameters stop updating without any error. The fix is a single call: `self.params = list(params)` in `__init__`, which eagerly consumes the generator and stores the resulting list for unlimited re-iteration.

## Worked solution

**Step 1 — Reproduce the bug.**
We build a `BuggyOpt` that stores `params` as-is. We pass a generator expression `(p for p in tensors)`. After the first `.step()` call, the generator is exhausted. The second `.step()` iterates an empty sequence and updates nothing.

**Step 2 — Verify the bug.**
We record the parameter values before and after each step. After step 1, values changed. After step 2, values are identical to step 1 — the bug is silent.

**Step 3 — Apply the fix.**
`FixedOpt.__init__` calls `self.params = list(params)`. The list can be iterated arbitrarily many times.

**Step 4 — Verify the fix.**
Both step 1 and step 2 update the parameters. The values after step 2 differ from values after step 1.

In [ ]:
import torch as t

class BuggyOpt:
    def __init__(self, params, lr):
        self.params = params   # BUG: raw generator stored
        self.lr = lr
    @t.no_grad()
    def step(self):
        for p in self.params:
            if p.grad is not None:
                p.data -= self.lr * p.grad

class FixedOpt:
    def __init__(self, params, lr):
        self.params = list(params)   # FIX: materialize
        self.lr = lr
    @t.no_grad()
    def step(self):
        for p in self.params:
            if p.grad is not None:
                p.data -= self.lr * p.grad
    def zero_grad(self):
        for p in self.params:
            p.grad = None

# --- exercise the bug ---
t.manual_seed(5)
a = t.tensor([3.0, -1.0], requires_grad=True)
a.grad = t.tensor([1.0, 1.0])

buggy = BuggyOpt((p for p in [a]), lr=0.1)
before = a.data.clone()
buggy.step()
after_1 = a.data.clone()
buggy.step()    # generator exhausted — no update
after_2 = a.data.clone()
print(f'Buggy: before={before}, after_1={after_1}, after_2={after_2}')
print(f'Step-2 changed anything: {not t.allclose(after_1, after_2)}')  # False!

# --- exercise the fix ---
t.manual_seed(5)
b = t.tensor([3.0, -1.0], requires_grad=True)
b.grad = t.tensor([1.0, 1.0])

fixed = FixedOpt([b], lr=0.1)
fixed.step()
val_1 = b.data.clone()
b.grad = t.tensor([1.0, 1.0])  # re-populate grad for step 2
fixed.step()
val_2 = b.data.clone()
print(f'Fixed: val_1={val_1}, val_2={val_2}')
print(f'Step-2 changed: {not t.allclose(val_1, val_2)}')  # True
assert not t.allclose(val_1, val_2), 'Fixed optimizer should update on step 2'